In [1]:
from datetime import datetime
from pathlib import Path
import re
import shutil
import pandas as pd
import pytesseract
from PIL import Image, ImageOps

# If needed: %pip install pandas pillow pytesseract openpyxl
# On macOS, install Tesseract first if missing: brew install tesseract

TESSERACT_CANDIDATES = [
    "/opt/homebrew/bin/tesseract",
    "/usr/local/bin/tesseract",
    "/opt/local/bin/tesseract",
    shutil.which("tesseract"),
]
for candidate in TESSERACT_CANDIDATES:
    if candidate and Path(candidate).exists():
        pytesseract.pytesseract.tesseract_cmd = candidate
        break


In [2]:
try:
    pytesseract.get_tesseract_version()
except Exception as exc:
    raise RuntimeError(
        "Tesseract OCR is not available. Install it first: brew install tesseract"
    ) from exc

ROOT_DIR = Path.cwd()
print("ROOT_DIR:", ROOT_DIR)
INPUT_DIR = ROOT_DIR / "cab_receipts"
OUTPUT_DIR = ROOT_DIR / "processed_receipts"
EXCEL_PATH = OUTPUT_DIR / "cab_receipts.xlsx"
SHEET_NAME = "Receipts"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "date", "amount", "receipt_number", "cab_period",
    "morning_cab", "evening_cab", "source_file"
]


ROOT_DIR: /Users/priyanshchaudhary/CodeUpdates/OCRReceipts


In [ ]:


def resolve_input_images():
    """Look for JPEG files in the project root or a cab_receipts folder."""
    locations = []
    if INPUT_DIR.exists() and INPUT_DIR.is_dir():
        locations.append(INPUT_DIR)
    locations.append(ROOT_DIR)

    seen = set()
    files = []
    for base_dir in locations:
        for image_path in sorted(base_dir.glob("*")):
            if image_path.is_file() and image_path.suffix.lower() in {".jpg", ".jpeg"}:
                resolved = image_path.resolve()
                if resolved not in seen:
                    files.append(image_path)
                    seen.add(resolved)
    return files


def extract_receipt_data(text):
    """Extract likely date, amount, receipt number, and time from OCR text."""
    cleaned_lines = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        line = line.replace("•", " ")
        line = line.replace("=", " ")
        line = re.sub(r"\s+", " ", line)
        cleaned_lines.append(line)

    lines = cleaned_lines

    # Find a date, supporting both numeric and text month formats from ride receipts.
    date_match = None
    date_patterns = [
        r"\b(\d{1,2})(?:st|nd|rd|th)?\s*(?:of\s+)?(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-zA-Z]*\s+(\d{4})\b",
        r"\b(\d{1,2})(?:st|nd|rd|th)?\s*(?:of\s+)?(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{4})\b",
        r"\b(\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\d{4}[/-]\d{1,2}[/-]\d{1,2})\b",
    ]

    month_names = {
        "jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
        "jul": 7, "aug": 8, "sep": 9, "sept": 9, "oct": 10, "nov": 11, "dec": 12,
        "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
        "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
    }

    for line in lines:
        for pattern in date_patterns:
            match = re.search(pattern, line, re.I)
            if not match:
                continue

            if len(match.groups()) == 3 and match.group(2).lower() in month_names:
                day = int(match.group(1))
                month = month_names[match.group(2).lower()]
                year = int(match.group(3))
                try:
                    date_match = datetime(year, month, day).date()
                    break
                except ValueError:
                    continue
            else:
                candidate = match.group(1)
                try:
                    parsed = pd.to_datetime(candidate, errors="coerce", dayfirst=True)
                    if pd.notna(parsed):
                        date_match = parsed.date()
                        break
                except Exception:
                    continue

        if date_match is not None:
            break

    # Prefer amounts on lines labeled total, fare, paid, or amount.
    amount = None
    amount_pattern = re.compile(
        r"(?:[₹Rr$%]|Rs\.?|INR)?\s*([0-9][0-9,]*(?:\.\d{1,2})?)",
        re.I,
    )
    amount_lines = [
        line for line in lines
        if re.search(r"\b(total|fare|paid|amount|suggested|ride charge)\b", line, re.I)
    ]
    for line in amount_lines + lines:
        match = amount_pattern.search(line)
        if match:
            try:
                value = match.group(1).replace(",", "")
                amount = float(value)
                if amount > 0:
                    break
            except ValueError:
                pass

    # Capture a ride or receipt identifier from common app receipt text.
    receipt_number = None
    for line in lines:
        if not re.search(r"\b(?:ride\s*id|receipt|trip|booking)\b", line, re.I):
            continue
        match = re.search(
            r"(?:ride\s*id|receipt|trip|booking)\s*(?:no\.?|number|id|#)?\s*[:#-]?\s*([A-Z0-9][A-Z0-9\s-]{4,})\b",
            line,
            re.I,
        )
        if match:
            receipt_number = re.sub(r"[^A-Z0-9]", "", match.group(1).upper())
            if len(receipt_number) >= 4:
                break

    # Prefer time values that include AM/PM, then fall back to the first valid time.
    time_match = None
    am_pm_lines = [line for line in lines if re.search(r"\b\d{1,2}:\d{2}\s*(AM|PM)\b", line, re.I)]
    for line in am_pm_lines + lines:
        match = re.search(r"\b(\d{1,2}):(\d{2})\s*(AM|PM)?\b", line, re.I)
        if not match:
            continue
        hour, minute = int(match.group(1)), int(match.group(2))
        suffix = (match.group(3) or "").upper()
        if minute < 60 and hour <= (12 if suffix else 23):
            if suffix:
                hour = hour % 12 + (12 if suffix == "PM" else 0)
            time_match = (hour, minute)
            if suffix:
                break

    return date_match, amount, receipt_number, time_match


def unique_destination(path):
    """Avoid overwriting a previously processed image."""
    if not path.exists():
        return path
    counter = 2
    while True:
        candidate = path.with_name(f"{path.stem}_{counter}{path.suffix}")
        if not candidate.exists():
            return candidate
        counter += 1


records = []
for image_path in resolve_input_images():
    if image_path.suffix.lower() not in {".jpg", ".jpeg"}:
        continue

    try:
        with Image.open(image_path) as opened_image:
            processed_image = ImageOps.grayscale(opened_image)
            processed_image = ImageOps.autocontrast(processed_image)
            text = pytesseract.image_to_string(processed_image)

        receipt_date, amount, receipt_number, receipt_time = extract_receipt_data(text)

        if receipt_date is None or amount is None:
            print(f"Skipped (date or amount not found): {image_path.name}")
            continue

        # Before 5 PM is classified as morning; 5 PM or later as evening.
        period = "morning" if receipt_time and receipt_time[0] < 17 else "evening"
        record = {
            "date": receipt_date.isoformat(),
            "amount": amount,
            "receipt_number": receipt_number,
            "cab_period": period,
            # "morning_cab": amount if period == "morning" else None,
            # "evening_cab": amount if period == "evening" else None,
            # "source_file": image_path.name,
        }
        records.append(record)

        receipt_suffix = f"_{receipt_number}" if receipt_number else ""
        filename = f"{receipt_date:%Y-%m-%d}_{amount}{period}{image_path.suffix.lower()}"
        # filename = f"{receipt_date:%Y-%m-%d}_{amount:.2f}{receipt_suffix}{image_path.suffix.lower()}"
        destination = unique_destination(OUTPUT_DIR / filename)
        shutil.move(str(image_path), str(destination))

    except Exception as exc:
        print(f"Could not process {image_path.name}: {exc}")

if records:
    new_data = pd.DataFrame(records, columns=COLUMNS)

    if EXCEL_PATH.exists():
        existing_data = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)
        new_data = pd.concat([existing_data, new_data], ignore_index=True)

    new_data.to_excel(EXCEL_PATH, sheet_name=SHEET_NAME, index=False)
    print(f"Saved {len(records)} receipt(s) to {EXCEL_PATH}")
else:
    print("No receipts were processed.")


No receipts were processed.
